In [ ]:
import cv2 as cv              # OpenCV计算机视觉库
import numpy as np            # NumPy数值计算库
import matplotlib.pyplot as plt  # Matplotlib绑图库

In [ ]:
def show(img):
    """自定义显示函数：自动判断灰度图或彩色图并正确显示"""
    if img.ndim == 2:  # 灰度图
        plt.imshow(img, cmap='gray')
    else:  # 彩色图，BGR转RGB
        img = cv.cvtColor(img, cv.COLOR_BGR2RGB)
        plt.imshow(img)
    plt.show()

# 1. Hough直线检测

## 1.1 Hough直线检测的OpenCV实现 

### 1.1.1 HoughLines

lines = cv2.HoughLines(image, rho, theta, threshold)
- lines:是一个输出参数，用于存储检测到的直线的参数。通常，这是一个二维数组，每一行代表一条直线，每行包含两个元素，分别是直线的 ρ（rho） 和 θ（theta）参数。
- image:这是输入的二值化图像，通常是Canny边缘检测的输出或其他二值化图像。
- rho  :表示霍夫空间中的距离分辨率（以像素为单位）。它决定了在霍夫空间中离散化的距离间隔大小。
- theta:表示霍夫空间中的角度分辨率（以弧度为单位）。它决定了在霍夫空间中离散化的角度间隔大小。
- threshold:这是一个阈值，表示检测直线所需的最小投票数。只有在霍夫空间中投票超过这个阈值的直线才会被检测到。

In [ ]:
bgr = cv.imread('pic/road200x200.jpg')  # 读取彩色道路图片
I   = cv.cvtColor(bgr, cv.COLOR_BGR2GRAY)  # 转为灰度图
E   = cv.Canny(I, 50, 400)  # Canny边缘检测，双阈值50和400

show(E)

In [ ]:
# HoughLines：标准霍夫直线检测，返回每条直线的(rho, theta)参数
lines = cv.HoughLines(E, 2, np.pi/180, 140)  # rho=2像素距离分辨率，theta=1度角度分辨率，threshold=140最小投票数
print(lines)       # 原始输出：三维数组 (N,1,2)
print(len(lines))  # 检测到的直线数量
print(lines.squeeze())  # 去掉多余维度，变为二维数组 (N,2)

In [ ]:
length = 1000  # 直线绘制的延伸长度

# 将HoughLines返回的极坐标参数转换为直线端点并绘制
for (rho, theta) in lines.squeeze():
    a = np.cos(theta)  # 方向向量的x分量
    b = np.sin(theta)  # 方向向量的y分量
    x0 = rho * a       # 原点到直线的垂足坐标
    y0 = rho * b

    # 从垂足沿垂直方向向两侧延伸得到两个端点
    x1 = int(x0 + length*(-b))
    y1 = int(y0 + length*(a))
    x2 = int(x0 - length*(-b))
    y2 = int(y0 - length*(a))
    
    cv.line(bgr, (x1, y1), (x2, y2), (0,0,255), 1)  # 在原图上绘制红色直线
    
show(bgr)

cv2.line(image, start_point, end_point, color, thickness)
- image:它是要在其上绘制线条的图像。
- start_point：它是线的起始坐标。坐标表示为两个值的元组，即(X坐标值，Y坐标值)。
- end_point：它是直线的终点坐标。坐标表示为两个值的元组，即(X坐标值ÿ坐标值)。
- color:它是要绘制的线条的颜色。对于BGR，我们通过一个元组。例如：(255，0，0)为蓝色。
- thickness:它是线的粗细像素。

### 1.1.2 HoughLinesP

lines = cv2.HoughLinesP(image, rho, theta, threshold, minLineLength, maxLineGap)
- minLineLength: 默认值为 0。最短线段的长度，比这个设定参数短的线段就不能被显现出来。
- maxLineGap: 默认值为 0。允许将同一行点与点之间连接起来的最大距离。

In [ ]:
bgr = cv.imread('pic/road200x200.jpg')  # 重新读取原图

# HoughLinesP：概率霍夫直线检测，直接返回线段端点坐标(x1,y1,x2,y2)
lines = cv.HoughLinesP(E, 2, np.pi/180, 80, minLineLength=20, maxLineGap=3)
# minLineLength=20：最短线段长度；maxLineGap=3：允许的最大间隙
for (x1, y1, x2, y2) in lines.squeeze():
    cv.line(bgr, (x1, y1), (x2, y2), (0, 0, 255), 1)
    
show(bgr)

##  1.2 Hough直线检测的编程实现

In [ ]:
# 手动实现霍夫直线检测（编程实现版）
I = cv.imread('pic/road200x200.jpg', 0)  # 读取灰度图
E = cv.Canny(I, 100, 400)  # Canny边缘检测
h, w = E.shape

dr = 2                    # rho距离分辨率（像素）
dt = np.pi / 180          # theta角度分辨率（1度=π/180弧度）
thresh = 130              # 投票阈值

# rho范围：0到对角线长度；theta范围：0到2π
rmin, rmax = 0, np.sqrt(h**2 + w**2)
tmin, tmax = 0, 2*np.pi

m = int((rmax - rmin) / dr) + 1  # 累加器rho维度
n = int((tmax - tmin) / dt) + 1  # 累加器theta维度
N = np.zeros((m,n), np.int32)    # 霍夫累加器，初始值为0

ys, xs = np.where(E == 255)  # 找到所有边缘像素坐标
thetas = np.arange(tmin, tmax, dt)  # theta采样序列

# 对每个边缘像素，遍历所有theta，计算对应的rho并投票
for (x,y) in zip(xs, ys):
    rhos = np.abs(x * np.cos(thetas) + y * np.sin(thetas))  # ρ = |x·cosθ + y·sinθ|
    ms = np.round(rhos / dr).astype(np.int32)  # 将rho映射到累加器行索引
    ns = np.round(thetas / dt).astype(np.int32) # 将theta映射到累加器列索引
    for (m,n) in zip(ms, ns):
        N[m,n] += 1  # 投票+1
        
show(N)

In [ ]:
plt.imshow(N, cmap='jet')  # 用jet颜色映射显示霍夫累加器，亮色区域表示投票多的(rho,theta)对
plt.show()

In [ ]:
bgr = cv.imread('pic/road200x200.jpg')  # 重新读取原图

rs, ts = np.where(N > thresh)  # 找到投票数超过阈值的累加器位置
rs = rs * dr   # 将索引转换回实际rho值
ts = ts * dt   # 将索引转换回实际theta值

length = 1000

# 将极坐标参数(rho, theta)转换为直线端点并绘制
for (rho, theta) in zip(rs, ts):
    a = np.cos(theta)
    b = np.sin(theta)
    x0 = rho * a
    y0 = rho * b
    x1 = int(x0 + length*(-b))
    y1 = int(y0 + length*(a))
    x2 = int(x0 - length*(-b))
    y2 = int(y0 - length*(a))
    
    cv.line(bgr, (x1, y1), (x2, y2), (0,0,255), 1)
    
show(bgr)

# 2.Hough圆检测

## 2.1 Hough圆检测的OpenCV实现

cv.HoughCircles(image, method, dp, minDist, param1, param2, minRadius, maxRadius)
- image: 灰度图（该函数里面会自动进行Canny边缘检测，检测阈值间 param1）
- method: 只提供霍夫梯度法， cv.HOUGH_GRADIENT, cv.HOUGH_GRADIENT_ALT
- dp: 图像分辨率与累加器分辨率之比，可取 dp=1~2
- minDist: 两个不同圆圆心之间的最小距离
- param1: 用于Canny的边缘阀值上限，下限被置为上限的一半
- param2: cv.HOUGH_GRADIENT 方法的累加器阈值，阈值越小，检测到的圆圈越多。
          选择 cv.HOUGH_GRADIENT_ALT 方法时，该参数取 0-1 之间，用于衡量圆的完美度，最完美时取 1.
- minRadius: 最小圆半径
- maxRadius: 最大圆半径

In [ ]:
bgr = cv.imread('pic/moons150x400.jpg')  # 读取月亮图片用于圆检测
I   = cv.cvtColor(bgr, cv.COLOR_BGR2GRAY)  # 转为灰度图

# HoughCircles：霍夫圆检测，函数内部会自动做Canny边缘检测
circles = cv.HoughCircles(I, cv.HOUGH_GRADIENT, dp=1.5, minDist=10, 
                param1=400, param2=50,        # param1=Canny高阈值，param2=累加器投票阈值
                minRadius=10, maxRadius=20)    # 圆半径范围10-20
circles  # 返回格式：[[[x, y, r], ...]]，(x,y)为圆心坐标，r为半径

In [ ]:
# 在原图上绘制检测到的圆
for (x,y,r) in circles.squeeze():
    cv.circle(bgr, (int(x), int(y)), int(r), (0,255,255), 1)  # 绘制黄色圆，线宽1
    
show(bgr)

In [ ]:
bgr = cv.imread('pic/moons150x400.jpg')
I   = cv.cvtColor(bgr, cv.COLOR_BGR2GRAY)

# HOUGH_GRADIENT_ALT：改进的霍夫梯度法，param2取0-1之间衡量圆的完美度
circles = cv.HoughCircles(I, cv.HOUGH_GRADIENT_ALT, dp=1.5, minDist=10,
                param1=400, param2=0.9,  # param2=0.9表示要求较高的圆完美度
                minRadius=10, maxRadius=20)

for (x,y,r) in circles.squeeze():
    cv.circle(bgr, (int(x), int(y)), int(r), (0,255,255), 1)

show(bgr)

## 2.2 Hough圆检测的编程实现

### 2.2.1 标准方法

In [ ]:
# 手动实现霍夫圆检测（标准方法）
bgr = cv.imread('pic/moons150x400.jpg')
I   = cv.cvtColor(bgr, cv.COLOR_BGR2GRAY)
E   = cv.Canny(I, 200, 400)  # Canny边缘检测
show(E)

h, w = E.shape

rmin, rmax, dr = 17, 19, 1   # 半径搜索范围和步长
nr = int((rmax - rmin) / dr) + 1

N = np.zeros((w, h, nr))      # 3D累加器：(宽, 高, 半径数)

ys, xs = np.where(E == 255)   # 边缘像素坐标
for (x,y) in zip(xs, ys):
    for r in range(rmin, rmax, dr):
        circ = np.zeros((h, w), np.uint8)
        cv.circle(circ, (x,y), r, 255, 1)  # 以边缘点为圆心画圆
        bs, aas = np.where(circ == 255)     # 圆上的所有点
        
        N[aas, bs, r-rmin] += 1  # 这些点作为圆心的投票+1

thresh = N.max() / 2             # 投票阈值取最大值的一半
aas, bs, rs = np.where(N >= thresh)

for (a,b,r) in zip(aas, bs, rs):
    cv.circle(bgr, (a,b), r+rmin, 255, 1)  # 绘制检测到的白色圆
    
show(bgr)

In [ ]:
plt.imshow(N[:,:,0].T, cmap='jet')  # 显示半径=rmin时的累加器切片，T转置以匹配图像坐标
plt.show()

In [ ]:
# 辅助演示：在空白图像上画一条指定角度的直线，帮助理解霍夫变换的几何原理
I = np.zeros((100,100), np.uint8)  # 100x100黑色图像
h, w = I.shape

theta = np.pi / 9     # 直线角度：20度（π/9弧度）
px = np.cos(theta)    # 方向向量x分量
py = np.sin(theta)    # 方向向量y分量

lmax = w + h          # 延伸长度足够覆盖整个图像
x, y = 50, 50         # 直线中心点
x1 = int(x + px*lmax)
y1 = int(y + py*lmax)
x2 = int(x - px*lmax)
y2 = int(y - py*lmax)

show(cv.line(I, (x1,y1), (x2,y2), 255, 1))  # 绘制白色直线

### 2.2.2 梯度法

In [ ]:
# 手动实现霍夫圆检测（梯度法）：利用边缘像素的梯度方向缩小圆心搜索范围
bgr = cv.imread('pic/moons150x400.jpg')
I   = cv.cvtColor(bgr, cv.COLOR_BGR2GRAY)
E   = cv.Canny(I, 200, 400)

h, w = E.shape

rmin, rmax, dr = 15, 20, 1

# 计算x和y方向的Sobel梯度
Ix = cv.Sobel(I, cv.CV_64F, 1, 0)
Iy = cv.Sobel(I, cv.CV_64F, 0, 1)
Ir = np.sqrt(Ix**2 + Iy**2)  # 梯度幅值
Ix /= (Ir + 1e-5)  # 归一化，避免除零，得到梯度方向单位向量
Iy /= (Ir + 1e-5)

N  = np.zeros((h, w), np.int32)  # 2D累加器（只累加圆心位置）
lmax = N.shape[0] + N.shape[1]

ys, xs = np.where(E == 255)
for (x, y) in zip(xs, ys):
    px = Ix[y, x]  # 边缘像素的梯度方向x分量
    py = Iy[y, x]  # 边缘像素的梯度方向y分量
    
    # 沿梯度方向画线，线上的点都可能作为圆心
    x1 = int(x + px*lmax)
    y1 = int(y + py*lmax)
    x2 = int(x - px*lmax)
    y2 = int(y - py*lmax)
    
    line = np.zeros(N.shape, np.uint8)
    cv.line(line, (x1, y1), (x2, y2), 255, 1)
    
    bs, aas = np.where(line == 255)
    N[bs, aas] += 1  # 梯度方向线上所有点的圆心投票+1

show(N)

In [ ]:
plt.imshow(N, cmap='jet')  # 用jet颜色映射显示梯度法的圆心累加器
plt.show()

In [ ]:
# 根据累加器找到候选圆心，再验证每个半径下的圆与边缘的重合度
thresh1 = N.max() / 2  # 圆心投票阈值
thresh2 = 30           # 圆与边缘重合的最小像素数

bs, aas = np.where(N >= thresh1)  # 候选圆心坐标
abr = []  # 存储通过验证的(a,b,r)三元组

for (a,b) in zip(aas, bs):
    for r in range(rmin, rmax, dr):
        circ = np.zeros(I.shape, np.uint8)
        cv.circle(circ, (a,b), r, 255, 1)  # 以候选圆心画圆
        intersection = cv.bitwise_and(circ, E)  # 与边缘图做与运算，取重合部分
        number = (intersection == 255).sum()    # 重合的边缘像素数
        if number > thresh2:  # 重合度足够高则确认为圆
            abr.append((a,b,r))

for (a,b,r) in abr:
    cv.circle(bgr, (a,b), r, (0,255,255), 1)
    
show(bgr)